# NeuroSLM — 3B Param Training on A100

Bio-inspired language model with MoE, adaptive compute, causal memory, and consciousness metrics.

**Training resumes automatically across runtime disconnects.**
Checkpoints + memory files are persisted to Git LFS.

| Preset | Params | GPU | VRAM | Batch |
|--------|--------|-----|------|-------|
| `large` | ~170M | T4 | 15GB | 4 |
| **`xl`** | **~3B** | **A100** | **40GB** | **2** |
| `xxl` | ~10B | 4×A100 | 160GB | 1 |

**Setup:** Runtime → Change runtime type → **A100** → Set `GITHUB_TOKEN` in cell 2

In [ ]:
# 1) GPU check — verify A100 is attached
import sys, torch, subprocess
print(f'Python {sys.version}')
subprocess.run(['nvidia-smi'])
print(f'\ntorch {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f'GPU: {name} ({vram:.1f} GB)')
    if vram < 35:
        print('⚠️ <40GB VRAM detected — use preset="large" (170M) instead of "xl" (3B)')
        print('   Switch to A100: Runtime → Change runtime type → A100')
    else:
        print('✓ Sufficient VRAM for 3B model')
else:
    print('❌ No GPU — enable GPU in Runtime → Change runtime type')

In [ ]:
# 2) Clone repo with PAT for push access
GITHUB_TOKEN = ""  # paste here, or add 'GITHUB_TOKEN' to Colab secrets

import os
if not GITHUB_TOKEN:
    try:
        from google.colab import userdata
        GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
        print("✓ Token loaded from Colab secrets")
    except Exception:
        print("⚠️ No token — checkpoints won't persist to Git")

REPO = f'https://{GITHUB_TOKEN}@github.com/269652/neuroslm.git' if GITHUB_TOKEN else 'https://github.com/269652/neuroslm.git'

if os.path.exists('neuroslm'):
    import shutil; shutil.rmtree('neuroslm')
!git clone {REPO}
%cd neuroslm
!git config user.email "train@neuroslm"
!git config user.name "NeuroSLM Trainer"

In [ ]:
# 3) Install deps + Git LFS + restore checkpoints & memory
!pip install -q --upgrade pip
!pip install -q -r requirements.txt 2>/dev/null || true
!apt-get install -y git-lfs -qq 2>/dev/null
!git lfs install
!git lfs pull

# Restore LFS checkpoints + memory files to working dir
import glob, shutil, os
os.makedirs('checkpoints', exist_ok=True)
restored = 0
for ext in ['*.pt', '*.mem', '*.mem.json']:
    for c in sorted(glob.glob(f'lfs_checkpoints/{ext}')):
        dest = os.path.join('checkpoints', os.path.basename(c))
        if not os.path.exists(dest):
            shutil.copy2(c, dest)
            print(f"  Restored: {os.path.basename(c)}")
            restored += 1

existing = sorted(glob.glob('checkpoints/*.pt'))
if existing:
    print(f"\n✓ {len(existing)} checkpoint(s) available. Latest: {existing[-1]}")
    # Show step info
    import torch
    ckpt = torch.load(existing[-1], map_location='cpu', weights_only=False)
    step = ckpt.get('step', '?')
    print(f"  Resumed at step {step}")
    del ckpt
else:
    print("No existing checkpoints — will train from scratch")

In [ ]:
# 4) Configuration — change these to control training
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

PRESET = "xl"            # "large" (170M, T4) | "xl" (3B, A100) | "xxl" (10B, multi-GPU)
TOTAL_STEPS = 100_000    # Target: 100K steps × batch_size × ctx = ~400M tokens per run
BATCH_SIZE = 2           # xl on A100: batch=2 fits; large on T4: batch=4
SAVE_EVERY = 1000        # Checkpoint + push frequency
MODE = "mix"             # "mix" = interleave text + chat (best for instruction following)
CHAT_RATIO = 0.6         # 60% chat/instruction, 40% text/knowledge

# Auto-detect: if GPU < 40GB VRAM, fall back to large preset
import torch
if torch.cuda.is_available():
    vram = torch.cuda.get_device_properties(0).total_mem / 1e9
    if vram < 35 and PRESET == "xl":
        print(f"⚠️ Only {vram:.0f}GB VRAM — falling back to large preset (170M)")
        PRESET = "large"
        BATCH_SIZE = 4
    else:
        print(f"✓ {vram:.0f}GB VRAM — using {PRESET} preset (batch={BATCH_SIZE})")

print(f"\nTraining plan:")
print(f"  Preset:     {PRESET}")
print(f"  Steps:      {TOTAL_STEPS:,}")
print(f"  Batch:      {BATCH_SIZE}")
print(f"  Mode:       {MODE} (chat_ratio={CHAT_RATIO})")
print(f"  Save every: {SAVE_EVERY}")

In [ ]:
# 5) Train — RESUMES AUTOMATICALLY from latest checkpoint
# Re-run this cell after every Colab reconnect to continue training.
# Training data: FineWeb-Edu + OpenHermes-2.5 + UltraChat + SlimOrca + more

!python -m neuroslm.train \
    --preset {PRESET} \
    --steps {TOTAL_STEPS} \
    --batch_size {BATCH_SIZE} \
    --device cuda \
    --meta \
    --mode {MODE} \
    --chat_ratio {CHAT_RATIO} \
    --save_every {SAVE_EVERY} \
    --resume latest

In [ ]:
# 6) Push checkpoint + memory to Git LFS
import glob, shutil, os

ckpts = sorted(glob.glob('checkpoints/*.pt'))
mems = sorted(glob.glob('checkpoints/*.mem'))
if ckpts:
    os.makedirs('lfs_checkpoints', exist_ok=True)
    pushed = []
    for f in ckpts[-2:] + mems[-2:]:  # push last 2 of each
        dest = os.path.join('lfs_checkpoints', os.path.basename(f))
        shutil.copy2(f, dest)
        pushed.append(os.path.basename(f))
    print(f"Pushing to Git LFS: {', '.join(pushed)}")
    !git add lfs_checkpoints/
    !git commit -m "chkpt: {pushed[-1] if pushed else 'update'}" 2>/dev/null || true
    !git push 2>/dev/null || echo "Push failed — check token"
    print("✓ Checkpoint + memory saved to Git LFS")
else:
    print("No checkpoints to push. Train first.")

In [ ]:
# 7) Benchmarks — HellaSwag, ARC-Easy, ARC-Challenge, MMLU
# Compares against SmolLM2, TinyLlama, Phi-3, Qwen2.5, Llama-3.1
import glob
ckpts = sorted(glob.glob('checkpoints/*.pt'))
if ckpts:
    latest = ckpts[-1]
    print(f"Benchmarking: {latest}")
    !python -m neuroslm.benchmarks --ckpt {latest} --preset {PRESET} --device cuda --max_samples 500
else:
    print("No checkpoints. Train first.")

In [ ]:
# 8) Generate text
import glob
ckpts = sorted(glob.glob('checkpoints/*.pt'))
if ckpts:
    latest = ckpts[-1]
    print(f'Using: {latest}')
    !python -m neuroslm.generate --ckpt {latest} --preset {PRESET} --prompt "Explain the theory of relativity in simple terms" --max_new 512 --temperature 0.7 --top_k 40 --device cuda
else:
    print('No checkpoints. Train first.')

In [ ]:
# 9) Intelligence Metrics — consciousness, identity drift, causal density
import glob, torch, sys
sys.path.insert(0, '.')
from neuroslm.config import PRESETS
from neuroslm.brain import Brain
from neuroslm.tokenizer import Tokenizer

ckpts = sorted(glob.glob('checkpoints/*.pt'))
assert ckpts, "No checkpoints. Train first."
latest = ckpts[-1]

tok = Tokenizer()
cfg = PRESETS[PRESET]()
cfg.vocab_size = tok.vocab_size
brain = Brain(cfg).to("cuda")
ckpt = torch.load(latest, map_location="cuda", weights_only=False)
brain.load_state_dict(ckpt["model"], strict=False)
brain.eval()
n_params = sum(p.numel() for p in brain.parameters())
print(f"Model: {n_params/1e6:.1f}M params, step {ckpt.get('step', '?')}")

# Load memory if available
import os
mem_path = latest.replace('.pt', '.mem')
if os.path.exists(mem_path):
    brain.load_memory_checkpoint(mem_path)
    print(f"Memory loaded: {mem_path}")

# Show intelligence metrics
brain.metrics.observe_narrative(brain.narrative_system)
brain.metrics.observe_memory(brain.episodic, brain.consolidated, brain.causal)
snap = brain.metrics.snapshot()
print("\n" + "=" * 60)
print("  INTELLIGENCE & CONSCIOUSNESS METRICS")
print("=" * 60)
for k, v in snap.items():
    if isinstance(v, float):
        print(f"  {k:30s}: {v:.4f}")
    else:
        print(f"  {k:30s}: {v}")
print("=" * 60)
print(f"\n  Causal rules learned: {brain.causal.stats()}")
print(f"  Memory gate stats: {brain.comprehension_gate.stats()}")

## Multi-day training workflow

1. **First run:** Run cells 1–6 sequentially. Training starts from scratch.
2. **Runtime disconnects:** Colab disconnects after ~12h. Checkpoints are auto-saved.
3. **Resume:** Re-run cells 1–5. Cell 5 auto-detects the latest checkpoint and continues.
4. **Track progress:** Run cell 7 (benchmarks) periodically to see improvement vs SOTA.
5. **Target:** 100K steps ≈ 400M tokens. For 10B tokens, increase `TOTAL_STEPS` to ~2.5M.

### Training data pipeline
- **Text:** FineWeb-Edu (10B tokens curated web), Cosmopedia, TinyStories
- **Chat:** OpenHermes-2.5, UltraChat-200k, WildChat-1M, SlimOrca, hh-rlhf, Dolly
- **Mode `mix`:** 60% chat + 40% text (tunable via `CHAT_RATIO`)

### Benchmark targets (3B xl preset)
| Benchmark | Random | Target | SOTA 3B (Qwen2.5) |
|-----------|--------|--------|-------------------|
| HellaSwag | 0.25 | >0.70 | 0.73 |
| ARC-Easy | 0.25 | >0.75 | 0.78 |
| ARC-Challenge | 0.25 | >0.50 | 0.53 |
| MMLU | 0.25 | >0.55 | 0.65 |